In [9]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets scikit-learn
!pip install -q -U --force-reinstall pyarrow

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Loading dataset + defining prompt and target

In [5]:
import pandas as pd
import json

df = pd.read_csv("/content/Stanza_AI_Intent_Classification_Final_700.csv")

def build_prompt(query):
    return f"""You are an intent classifier for a hotel AI system called Stanza AI. Given a guest or staff query, identify the responsible agent and the specific intent.

Query: {query}
Answer (JSON only):"""

def build_target(agent_label, intent_name):
    return json.dumps({"agent_label": agent_label, "intent_name": intent_name})

df["prompt"] = df["query"].apply(build_prompt)
df["target"] = df.apply(lambda r: build_target(r["agent_label"], r["intent_name"]), axis=1)
df["full_text"] = df["prompt"] + " " + df["target"]

print(df[["query", "prompt", "target"]].iloc[0]["prompt"])
print()
print("Target:", df.iloc[0]["target"])

You are an intent classifier for a hotel AI system called Stanza AI. Given a guest or staff query, identify the responsible agent and the specific intent.

Query: Can I add a early check-in package to my reservation?
Answer (JSON only):

Target: {"agent_label": "reservations", "intent_name": "reservation_upsell"}


Train/Validation/Test Split

In [6]:
from datasets import Dataset

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

train_dataset = Dataset.from_pandas(train_df[["full_text"]])
val_dataset = Dataset.from_pandas(val_df[["prompt", "target", "agent_label", "intent_name"]])
test_dataset = Dataset.from_pandas(test_df[["prompt", "target", "agent_label", "intent_name"]])

print(train_dataset)

Train size: 560
Validation size: 70
Test size: 70
Dataset({
    features: ['full_text'],
    num_rows: 560
})


Phi-4-mini Base Model Load (4-bit Quantized)

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "microsoft/Phi-4-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded:", MODEL_NAME)
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Model loaded: microsoft/Phi-4-mini-instruct
GPU memory used (GB): 2.89


LoRA Setup

In [12]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 3,839,167,488 || trainable%: 0.0819


Training

In [13]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./phi-stanza-finetune",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    max_length=512,
    dataset_text_field="full_text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=Dataset.from_pandas(val_df[["full_text"]]),
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/70 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/70 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/70 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.503575,0.491145,0.471620,37224.000000,0.890697
2,0.391969,0.413545,0.391788,74448.000000,0.910774
3,0.359438,0.394237,0.356167,111672.000000,0.915557


TrainOutput(global_step=210, training_loss=0.6513829253968738, metrics={'train_runtime': 1997.4999, 'train_samples_per_second': 0.841, 'train_steps_per_second': 0.105, 'total_flos': 2217640580444160.0, 'train_loss': 0.6513829253968738, 'epoch': 3.0})

Drive Backup

In [14]:
import shutil, os
print("Checkpoints available:", os.listdir("./phi-stanza-finetune"))

Checkpoints available: ['checkpoint-210', 'checkpoint-140', 'checkpoint-70', 'README.md']


In [15]:
shutil.copytree(
    "./phi-stanza-finetune/checkpoint-210",   # yaha sahi naam daalna
    "/content/drive/MyDrive/stanza_ai_phi_checkpoint",
    dirs_exist_ok=True
)
print("Phi backed up to Google Drive!")

Phi backed up to Google Drive!


Evaluation on test dataset

In [16]:
import re, json as jsonlib
from tqdm import tqdm

model.eval()

def generate_answer(prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip()

def extract_json(text):
    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if match:
        try:
            return jsonlib.loads(match.group())
        except:
            return None
    return None

results = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    raw_output = generate_answer(row["prompt"])
    parsed = extract_json(raw_output)
    results.append({
        "true_agent": row["agent_label"],
        "true_intent": row["intent_name"],
        "pred_agent": parsed.get("agent_label") if parsed else None,
        "pred_intent": parsed.get("intent_name") if parsed else None,
        "raw_output": raw_output,
    })

phi_results_df = pd.DataFrame(results)
phi_results_df.head(10)

100%|██████████| 70/70 [02:09<00:00,  1.85s/it]


,true_agent,true_intent,pred_agent,pred_intent,raw_output
0,reservations,reservation_cancel,reservations,reservation_cancel,"{""agent_label"": ""reservations"", ""intent_name"":..."
1,reservations,reservation_create,reservations,reservation_create,"{""agent_label"": ""reservations"", ""intent_name"":..."
2,reservations,reservation_availability,reservations,reservation_availability,"{""agent_label"": ""reservations"", ""intent_name"":..."
3,reservations,reservation_rate_question,reservations,reservation_rate_question,"{""agent_label"": ""reservations"", ""intent_name"":..."
4,reservations,reservation_cancel,reservations,reservation_cancel,"{""agent_label"": ""reservations"", ""intent_name"":..."
5,reservations,reservation_rate_question,reservations,reservation_rate_question,"{""agent_label"": ""reservations"", ""intent_name"":..."
6,reservations,reservation_modify,reservations,reservation_modify,"{""agent_label"": ""reservations"", ""intent_name"":..."
7,reservations,reservation_create,reservations,reservation_create,"{""agent_label"": ""reservations"", ""intent_name"":..."
8,reservations,reservation_upsell,reservations,reservation_upsell,"{""agent_label"": ""reservations"", ""intent_name"":..."
9,reservations,reservation_availability,reservations,reservation_availability,"{""agent_label"": ""reservations"", ""intent_name"":..."


saving result to drives

In [17]:
phi_results_df.to_csv("/content/drive/MyDrive/phi_evaluation_results.csv", index=False)
print("Evaluation results saved to Drive!")

Evaluation results saved to Drive!


calculating accuracy

In [18]:
phi_results_df["fully_correct"] = (phi_results_df["pred_agent"] == phi_results_df["true_agent"]) & (phi_results_df["pred_intent"] == phi_results_df["true_intent"])
overall = phi_results_df["fully_correct"].mean() * 100
agent_acc = (phi_results_df["pred_agent"] == phi_results_df["true_agent"]).mean() * 100
intent_acc = (phi_results_df["pred_intent"] == phi_results_df["true_intent"]).mean() * 100

print(f"PHI-4-mini — Overall: {overall:.2f}% | Agent: {agent_acc:.2f}% | Intent: {intent_acc:.2f}%")

PHI-4-mini — Overall: 78.57% | Agent: 92.86% | Intent: 81.43%
